In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType


#Define expected schema

schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("email", StringType(), True),
    StructField(
        "profile",
        StructType([
            StructField("age", IntegerType(), True),
            StructField("gender", StringType(), True),
            StructField(
                "location",
                StructType([
                    StructField("city", StringType(), True),
                    StructField("state", StringType(), True),
                    StructField("country", StringType(), True)
                ]),
                True
            )
        ]),
        True
    )
])



df = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "json")
    .option("cloudfiles.schemaLocation", "/Volumes/retailnew/bronze/schema_logs/customer")
    .schema(schema)
    .load("/Volumes/retailnew/bronze/raw_data/customers")
    .withColumn("ingest_ts", F.current_timestamp())
       
)
(
    df.writeStream
    .format("Delta")
    .option("checkpointLocation", "/Volumes/retailnew/bronze/checkpoints/customer")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table("retailnew.bronze.customers")
)